# Day 3 - Seller Trust Score

A score from 0 to 100 that ranks each Olist seller by customer-trust performance. Higher is better.

The score combines four components, weighted by what the Day 2 hypotheses showed mattered most:

| Component | Weight |
|---|---|
| Route-adjusted late delivery rate | 50% |
| Mean review score | 25% |
| One-star review rate | 15% |
| Category quality penalty (office_furniture) | 10% |

Each metric is converted to the seller's percentile rank within the seller population. The trust score is the weighted average of those ranks. Minimum 10 orders required to receive a score.

H1 (`02_h1_review_drivers.ipynb`) showed delivery timeliness drives 2.48x more review variance than category does, which is why delivery has 50% weight. H2 (`03_h2_geographic.ipynb`) showed late delivery is extremely concentrated in specific routes, so a seller's late rate must be benchmarked against the routes they actually serve, not the global platform average.

This notebook:
1. Per-seller raw metrics + volume filter
2. Build the trust score
3. Validate (distribution, top/bottom 10, correlation check)
4. Business impact (Pareto: bottom X% of sellers drive Y% of one-star reviews)
5. Findings going into Day 4


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.load import load_analysis_df

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
sns.set_palette("Set2")

analysis_df = load_analysis_df()
print(f"analysis_df shape: {analysis_df.shape}")
print(f"unique sellers: {analysis_df['seller_id'].nunique():,}")


analysis_df shape: (112650, 28)
unique sellers: 3,095


## 1. Per-seller raw metrics

For each seller, compute the raw metrics that go into the trust score. Computed at the order level (multi-item orders count once) with proper denominators:

- `n_orders`: total order count
- `late_rate`: % of delivered orders that arrived after the estimated date
- `mean_review`: mean review score among reviewed orders
- `one_star_rate`: % of reviewed orders that got 1 star
- `primary_category`: most-frequent category for that seller
- `is_office_furniture`: True if primary category is office_furniture (the H1 falsification exception)


In [2]:
# Order-level view (one row per order, since trust score is about orders not items).
order_level = analysis_df.drop_duplicates("order_id").copy()
order_level["is_late"] = order_level["promise_gap_days"] < 0
order_level["is_one_star"] = order_level["review_score"] == 1

# Aggregate per seller.
seller_metrics = (order_level.groupby("seller_id").agg(
    n_orders=("order_id", "count"),
    delivered=("order_delivered_customer_date", "count"),
    late_count=("is_late", "sum"),
    reviewed=("review_score", "count"),
    one_star_count=("is_one_star", "sum"),
    mean_review=("review_score", "mean"),
).reset_index())

seller_metrics["late_rate"] = seller_metrics["late_count"] / seller_metrics["delivered"]
seller_metrics["one_star_rate"] = seller_metrics["one_star_count"] / seller_metrics["reviewed"]

# Primary category per seller (the category they sell most often).
primary_cat = (analysis_df.groupby(["seller_id", "category_en"])
               .size()
               .reset_index(name="n")
               .sort_values(["seller_id", "n"], ascending=[True, False])
               .drop_duplicates("seller_id", keep="first")
               [["seller_id", "category_en"]]
               .rename(columns={"category_en": "primary_category"}))

seller_metrics = seller_metrics.merge(primary_cat, on="seller_id", how="left")
seller_metrics["is_office_furniture"] = seller_metrics["primary_category"] == "office_furniture"

print(f"Total sellers in seller_metrics: {len(seller_metrics):,}")
print(f"Sellers with >=10 orders:        {(seller_metrics['n_orders'] >= 10).sum():,}")
print(f"Sellers in office_furniture:     {seller_metrics['is_office_furniture'].sum():,}")
print()
print("Sample of seller_metrics (first 5 rows):")
seller_metrics.head()


Total sellers in seller_metrics: 3,088
Sellers with >=10 orders:        1,258
Sellers in office_furniture:     17

Sample of seller_metrics (first 5 rows):


,seller_id,n_orders,delivered,late_count,reviewed,one_star_count,mean_review,late_rate,one_star_rate,primary_category,is_office_furniture
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,0,3,1,3.666667,0.000000,0.333333,small_appliances,False
1,001cca7ae9ae17fb1caed9dfb1094831,199,194,13,196,23,4.000000,0.067010,0.117347,garden_tools,False
2,001e6ad469a905060d959994f1b41e4f,1,0,0,1,1,1.000000,NaN,1.000000,sports_leisure,False
3,002100f778ceb8431b7a1020ff7ab48f,50,49,9,50,7,3.880000,0.183673,0.140000,furniture_decor,False
4,003554e2dce176b5555353e4f3555ac8,1,1,0,1,0,5.000000,0.000000,0.000000,unknown,False


## 2. Route-adjusted late rate

A seller in SP shipping to AL is on a structurally hard route (26.3% late per H2). They shouldn't be penalised for being on that route. Conversely, a seller on an easy route who is still late is doing worse than expected.

The route adjustment computes, for each seller, the excess late rate over what's expected given the routes they serve:

1. For every (seller_state, customer_state) route, compute the median late rate across all sellers on that route. This is the "route baseline".
2. For each seller, compute their expected late rate as the volume-weighted average of route baselines for the routes they actually serve.
3. `route_excess_late` = seller's actual late rate minus their expected late rate.

Positive `route_excess_late` means the seller is performing worse than expected for their route mix. Negative means better than expected.


In [3]:
# Step 1: Route baseline late rate (across all sellers on each route).
route_baseline = (order_level
                  .dropna(subset=["seller_state", "customer_state", "promise_gap_days"])
                  .groupby(["seller_state", "customer_state"])
                  .agg(route_late_rate=("is_late", "mean"))
                  .reset_index())

# Step 2: Per-seller, per-route order count + late rate.
seller_route = (order_level
                .dropna(subset=["seller_state", "customer_state", "promise_gap_days"])
                .groupby(["seller_id", "seller_state", "customer_state"])
                .agg(n=("order_id", "count"))
                .reset_index()
                .merge(route_baseline, on=["seller_state", "customer_state"], how="left"))

# Step 3: Volume-weighted expected late rate per seller.
seller_route["weight_x_baseline"] = seller_route["n"] * seller_route["route_late_rate"]
expected = (seller_route.groupby("seller_id")
            .agg(total_route_orders=("n", "sum"),
                 sum_weighted_baseline=("weight_x_baseline", "sum"))
            .reset_index())
expected["expected_late_rate"] = (
    expected["sum_weighted_baseline"] / expected["total_route_orders"]
)

# Merge expected back into seller_metrics; route_excess_late = actual - expected.
seller_metrics = seller_metrics.merge(
    expected[["seller_id", "expected_late_rate"]],
    on="seller_id",
    how="left",
)
seller_metrics["route_excess_late"] = (
    seller_metrics["late_rate"] - seller_metrics["expected_late_rate"]
)

print(f"Routes with baseline computed: {len(route_baseline):,}")
print(f"Sellers with expected_late_rate: {seller_metrics['expected_late_rate'].notna().sum():,}")
print()
print("route_excess_late distribution (positive = worse than expected for the seller's route mix):")
print(seller_metrics["route_excess_late"].describe().round(4))


Routes with baseline computed: 409
Sellers with expected_late_rate: 2,960

route_excess_late distribution (positive = worse than expected for the seller's route mix):
count    2960.0000
mean        0.0088
std         0.1687
min        -0.2627
25%        -0.0690
50%        -0.0422
75%         0.0237
max         0.9787
Name: route_excess_late, dtype: float64


## 3. Volume filter

Apply the minimum-volume filter (at least 10 orders) so the trust score is based on a meaningful sample. Below 10 orders, a single bad review or one late delivery distorts every metric.

Some metrics need additional non-null requirements: `late_rate` needs delivered orders, `mean_review` and `one_star_rate` need reviewed orders. Phase 2 (the score build) will require all of them to be present.


In [4]:
scoreable = seller_metrics[seller_metrics["n_orders"] >= 10].copy()

total_orders_all = seller_metrics["n_orders"].sum()
total_orders_eligible = scoreable["n_orders"].sum()

print(f"Total sellers in dataset:                    {len(seller_metrics):,}")
print(f"Sellers eligible for trust score (>=10 orders): {len(scoreable):,} ({len(scoreable)/len(seller_metrics):.1%})")
print(f"Order coverage by eligible sellers:           {total_orders_eligible:,} of {total_orders_all:,} ({total_orders_eligible/total_orders_all:.1%})")
print()
print("Summary stats for eligible sellers' raw metrics:")
scoreable[["n_orders", "late_rate", "mean_review", "one_star_rate", "route_excess_late"]].describe().round(3)


Total sellers in dataset:                    3,088
Sellers eligible for trust score (>=10 orders): 1,258 (40.7%)
Order coverage by eligible sellers:           92,556 of 98,666 (93.8%)

Summary stats for eligible sellers' raw metrics:


,n_orders,late_rate,mean_review,one_star_rate,route_excess_late
count,1258.000,1258.000,1258.000,1258.000,1258.000
mean,73.574,0.081,4.130,0.110,0.003
std,153.870,0.075,0.405,0.092,0.072
min,10.000,0.000,1.263,0.000,-0.126
25%,16.000,0.029,3.933,0.056,-0.043
50%,30.000,0.069,4.182,0.095,-0.010
75%,67.000,0.114,4.391,0.145,0.033
max,1844.000,0.643,5.000,0.895,0.552
